## Collecting data about companies by web-scraping from ambition-box

### 7 features -- 

- Company name
- Company rating
- No of reviews
- Company Type
- Headquarter Location
- Critically Rated for
- Highly Rated for

In [128]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [129]:
#### NOTE:

#Error -- 403, means that the server has rejected the request
#Ambition box has a robots.txt file created internally to prevent web-scraping
#So, request needs to be disguised in a manner that ambition box thinks that request is coming from browser by a human and not from a bot

#To disguise the request to look like its a human from a browser, use headers. create a headers dict with value = user agent(can be found by
#searching My user agent on google), key = 'User-Agent'. Then pass the headers dict as value of parameter - headers, in requests.get

#This goes to the url and gets its whole html code from page source!

In [130]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'}
webpage = requests.get("https://www.ambitionbox.com/list-of-companies?page=1", headers=headers).text

In [131]:
#Now create a soup object, and give it the webpage along with the built in HTML parser (lxml)
soup = BeautifulSoup(webpage, 'lxml')
# print(soup.prettify()) #It makes the html more organized and easy to understand

In [132]:
#To print all the h1 tags in webpage--

soup.find_all('h1') #got list as an o/p, there is only one h1 tag in webpage

[<h1 class="companyListing__collectionTopHeadingCont" data-v-bb58b91c=""><div class="companyListing__collectionTopHeading" data-v-bb58b91c="">
 					Top Companies in
 				</div> <div class="companyListing__collectionIndiaHeading" data-v-bb58b91c=""><img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/>
 					INDIA
 					<img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/></div></h1>]

In [133]:
soup.find_all('h1')[0]

<h1 class="companyListing__collectionTopHeadingCont" data-v-bb58b91c=""><div class="companyListing__collectionTopHeading" data-v-bb58b91c="">
					Top Companies in
				</div> <div class="companyListing__collectionIndiaHeading" data-v-bb58b91c=""><img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/>
					INDIA
					<img data-v-bb58b91c="" height="12" src="https://static.ambitionbox.com/static/loc/loc-star.png" width="12"/></div></h1>

In [134]:
#To get the text of h1 tag
soup.find_all('h1')[0].text

'\n\t\t\t\t\tTop Companies in\n\t\t\t\t \n\t\t\t\t\tINDIA\n\t\t\t\t\t'

In [135]:
soup.find_all('h2')

[<h2 class="companyListing__title">
 								Companies in India
 							</h2>,
 <h2 class="companyCardWrapper__companyName" title="TCS">
 									TCS
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Accenture">
 									Accenture
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Wipro">
 									Wipro
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Cognizant">
 									Cognizant
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Capgemini">
 									Capgemini
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="HDFC Bank">
 									HDFC Bank
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="Infosys">
 									Infosys
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="HCLTech">
 									HCLTech
 								</h2>,
 <h2 class="companyCardWrapper__companyName" title="ICICI Bank">
 									ICICI Bank
 								</h2>,
 <h2 class="companyCardWrapper__companyName" ti

In [136]:
# soup.find_all('h2')[0].text.strip() ## -- Printing all company names using a for loop

for i in soup.find_all('h2'):
    print(i.text.strip()) ##Company names scraped from webpage-1

Companies in India
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services
Popular Collections by Industries
Popular Collections by Cities
Popular Collections by Roles


In [137]:
# Now along with company names, there are also some extra outputs like Popular Collections by Industries etc..
# use class attribute of soup.find_all

soup.find_all('h2', class_= 'companyCardWrapper__companyName') #see the class which the h2 tag belongs too

print(len(soup.find_all('h2', class_= 'companyCardWrapper__companyName'))) #20 companies in 1 webpage

for i in soup.find_all('h2', class_= 'companyCardWrapper__companyName'):
    print(i.text.strip())


20
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services


In [138]:
# Similarly we can extract ratings and reviews, but if we try to extract headquarter location and type of company,
# We can see on the html code of webpage1 that they have the same class!
# So, to fix this issue instead of going feature by feature through many loops, i will first extract the individual company cards
# and then will loop inside that company card

In [139]:
# We will first extract company cards and then we will loop inside it..
# The features which i need are present in the primary info div --

soup.find_all('div', class_= 'companyCardWrapper__primaryInformation')[0]

# This is the container for the first company, contains all the features i need for dataset

<div class="companyCardWrapper__primaryInformation"><div class="companyCardWrapper__companyLogo"><img alt="Tata Consultancy Services logo" height="50" loading="lazy" onerror="this.onerror=null;this.src='/static/icons/company-placeholder.svg';" src="https://static.ambitionbox.com/assets/v2/images/rs:fit:200:200:false:false/aHR0cHM6Ly9tZWRpYS5uYXVrcmkuY29tL21lZGlhL2FiY29tcGxvZ28vdGNzLW9yaWdpbmFsLmpwZw.webp" width="50"/></div> <div class="companyCardWrapper__metaInformation"><div class="companyCardWrapper__header"><div class="companyCardWrapper__companyPrimaryDetailsTopSection"><a class="companyCardWrapper__companyName"><h2 class="companyCardWrapper__companyName" title="TCS">
									TCS
								</h2></a> <!-- --></div> <button arialabel="Follow" class="companyCardWrapper__FollowCTA g-btn g-btn--text g-btn--md" data-testid="" title="Follow" type="button"><!-- --> <span class="g-btn__label g-btn__label--md g-btn__label--text g-btn__label--loader">Follow</span> <!-- --> <!-- --></button> <

In [ ]:
names = []
ratings = []
num_reviews = []
high_rated = []
critical_rated = []

for i in soup.find_all('div', class_= 'companyCardWrapper__primaryInformation'):

    names.append(i.find('h2', class_= 'companyCardWrapper__companyName').text.strip())
    ratings.append(i.find('div', class_='rating_text').text.strip())
    num_reviews.append(i.find('span', class_='companyCardWrapper__companyRatingCount').text.strip())


#Code to extract highly rated for and critically rated for
    ratings = i.find_all('span', class_= 'companyCardWrapper__ratingValues')

    high = None
    critical = None

    for span in ratings:

        parent_text = span.parent.text.upper()

        if 'HIGHLY RATED' in parent_text:
            high = span.text.strip()
        elif 'CRITICALLY RATED' in parent_text:
            critical = span.text.strip()
    high_rated.append(high)
    critical_rated.append(critical)



print(names)
print(ratings)
print(num_reviews)
print(high_rated)
print(critical_rated)

# names, ratings, num_reviews, high_rated and critical_rated of each company successfully extracted from webpage1

['TCS', 'Accenture', 'Wipro', 'Cognizant', 'Capgemini', 'HDFC Bank', 'Infosys', 'HCLTech', 'ICICI Bank', 'Tech Mahindra', 'Genpact', 'TP', 'Axis Bank', 'Jio', 'Concentrix Corporation', 'Amazon', 'Reliance Retail', 'iEnergizer', 'LTM Limited', 'HDB Financial Services']
[<span class="companyCardWrapper__ratingValues">Promotions, Salary</span>]
['(1.2L)', '(75.2k)', '(66.4k)', '(62.7k)', '(54.8k)', '(53.9k)', '(49.9k)', '(47k)', '(46.7k)', '(44.2k)', '(43.4k)', '(39.7k)', '(34.1k)', '(34.1k)', '(32.9k)', '(32.4k)', '(27.8k)', '(27.5k)', '(27k)', '(26.3k)']
['Job Security', None, None, None, 'Work Life Balance, Job Security', 'Job Security, Skill Development', 'Job Security', None, 'Job Security, Promotions, Skill Development', None, 'Job Security', 'Work Life Balance, Job Security, Skill Development', None, 'Job Security, Skill Development, Company Culture', 'Job Security', 'Company Culture, Work Life Balance, Salary', 'Job Security, Skill Development, Work Life Balance', 'Company Culture

In [142]:
# Difference between soup.find and soup.find_all--

### -- soup.find is used when we only have one tag with same class..
### -- soup.find_all is used when we have multiple tags with same class.. so we get a list [look at the loop above]